<a href="https://colab.research.google.com/github/toni2174301/SportB/blob/main/SportB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pyomo.environ as pyo

# Create a concrete model
model = pyo.ConcreteModel()

# --- Sets ---
# Define a set of warehouses
model.Warehouses = pyo.Set(initialize=['W1', 'W2', 'W3', 'W4'])

# Define a set for potential delivery routes between warehouses (all distinct pairs)
model.Routes = pyo.Set(initialize=model.Warehouses * model.Warehouses, filter=lambda model, i, j: i != j)

# --- Parameters (Example Data - these would typically be loaded from external sources) ---
# Base physical distance between warehouses (a given parameter)
# We will use an 'effective_distance_factor' as a decision variable to influence this.
model.base_physical_distance = pyo.Param(model.Routes, initialize={
    ('W1', 'W2'): 100, ('W1', 'W3'): 150, ('W1', 'W4'): 200,
    ('W2', 'W1'): 100, ('W2', 'W3'): 80, ('W2', 'W4'): 120,
    ('W3', 'W1'): 150, ('W3', 'W2'): 80, ('W3', 'W4'): 90,
    ('W4', 'W1'): 200, ('W4', 'W2'): 120, ('W4', 'W3'): 90,
})

# Example demand at each warehouse (can be 0 for source warehouses)
model.demand = pyo.Param(model.Warehouses, initialize={'W1': 0, 'W2': 50, 'W3': 70, 'W4': 80})

# Example initial capacity of each warehouse (can be 0 for pure demand warehouses)
model.capacity = pyo.Param(model.Warehouses, initialize={'W1': 200, 'W2': 0, 'W3': 0, 'W4': 0})

# Cost factors for the decision variables and overall objective
# Cost per unit of flow over effective distance
model.cost_per_unit_flow_per_distance = pyo.Param(initialize=0.05)

# Cost to "improve" distance (e.g., invest in better routes or infrastructure)
# A lower `effective_distance_factor` (meaning shorter effective distance) incurs a higher fixed cost.
model.cost_to_improve_distance_per_unit = pyo.Param(initialize=50)

# Cost associated with increasing delivery speed (e.g., faster transport costs more per unit of speed increase)
model.cost_per_unit_speed_increase = pyo.Param(initialize=10)

# How much capacity/efficiency increases per unit increase in running cost above the minimum
model.efficiency_gain_per_running_cost_unit = pyo.Param(initialize=0.01)

# Bounds for decision variables
# effective_distance_factor: Can be between 0.7 (30% reduction from base_physical_distance) and 1.0 (no reduction)
model.min_effective_distance_factor = pyo.Param(initialize=0.7)
model.max_effective_distance_factor = pyo.Param(initialize=1.0)

# delivery_speed: Can be between 20 and 100 (e.g., km/h or arbitrary units)
model.min_delivery_speed = pyo.Param(initialize=20)
model.max_delivery_speed = pyo.Param(initialize=100)

# warehouse_running_cost: Can be between 500 and 2000 (e.g., monetary units)
model.min_warehouse_running_cost = pyo.Param(initialize=500)
model.max_warehouse_running_cost = pyo.Param(initialize=2000)

# --- Decision Variables ---
# 1. flow[i, j]: Amount of goods delivered from warehouse i to warehouse j
#    Domain: NonNegativeReals (cannot be negative)
model.flow = pyo.Var(model.Routes, domain=pyo.NonNegativeReals)

# 2. effective_distance_factor[i, j]: A factor applied to base_physical_distance.
#    A lower factor means a more 'efficient' or 'shorter' effective distance.
#    This variable represents the 'distance between warehouses' as a decision variable.
#    Initialized to 1.0 (no reduction) to show default state if no improvement is chosen.
model.effective_distance_factor = pyo.Var(model.Routes,
                                          bounds=(model.min_effective_distance_factor, model.max_effective_distance_factor),
                                          initialize=1.0)

# 3. warehouse_running_cost[w]: The chosen level of running cost for each warehouse.
#    This directly contributes to the total cost and can influence capacity/efficiency.
#    This represents 'cost of running each warehouse' as a decision variable.
#    Initialized to its minimum.
model.warehouse_running_cost = pyo.Var(model.Warehouses,
                                      bounds=(model.min_warehouse_running_cost, model.max_warehouse_running_cost),
                                      initialize=model.min_warehouse_running_cost)

# 4. delivery_speed[i, j]: The chosen speed for deliveries between warehouses.
#    Higher speed contributes more to the cost.
#    This represents 'speed of delivering between warehouses' as a decision variable.
#    Initialized to its minimum.
model.delivery_speed = pyo.Var(model.Routes,
                                bounds=(model.min_delivery_speed, model.max_delivery_speed),
                                initialize=model.min_delivery_speed)

# --- Objective Function: Minimize Total Cost ---
# Total Cost = Delivery Flow Cost + Distance Optimization Cost + Speed Optimization Cost + Warehouse Running Cost
def total_cost_rule(model):
    # 1. Cost associated with the actual flow of goods based on effective distance
    flow_delivery_cost = sum(
        model.flow[i, j] * model.base_physical_distance[i, j] * model.effective_distance_factor[i, j] * model.cost_per_unit_flow_per_distance
        for i, j in model.Routes
    )

    # 2. Cost associated with choosing to improve the effective distance (i.e., making effective_distance_factor lower)
    #    The cost increases as the factor moves from its maximum (no improvement) towards its minimum (most improvement)
    distance_optimization_cost = sum(
        (model.max_effective_distance_factor - model.effective_distance_factor[i, j]) * model.cost_to_improve_distance_per_unit
        for i, j in model.Routes
    )

    # 3. Cost associated with chosen delivery speed (faster speed costs more)
    speed_optimization_cost = sum(
        (model.delivery_speed[i, j] - model.min_delivery_speed) * model.cost_per_unit_speed_increase
        for i, j in model.Routes
    )

    # 4. Direct cost of running warehouses (this is the decision variable itself)
    warehouse_total_running_cost = sum(model.warehouse_running_cost[w] for w in model.Warehouses)

    return flow_delivery_cost + distance_optimization_cost + speed_optimization_cost + warehouse_total_running_cost

model.objective = pyo.Objective(rule=total_cost_rule, sense=pyo.minimize)

# --- Constraints ---
# 1. Supply constraints: Total outflow from a warehouse cannot exceed its capacity.
#    Capacity can be influenced by the chosen `warehouse_running_cost`.
def supply_constraint_rule(model, i):
    # Dynamic capacity: initial capacity + (investment in running cost above minimum * efficiency gain)
    dynamic_capacity = model.capacity[i] + \
                       (model.warehouse_running_cost[i] - model.min_warehouse_running_cost) * model.efficiency_gain_per_running_cost_unit

    # The constraint is always valid and will enforce `sum(...) <= 0` if dynamic_capacity is 0
    return sum(model.flow[i, j] for j in model.Warehouses if (i, j) in model.Routes) <= dynamic_capacity
model.supply_constraint = pyo.Constraint(model.Warehouses, rule=supply_constraint_rule)

# 2. Demand constraints: Total inflow to a warehouse must meet its demand.
def demand_constraint_rule(model, j):
    if model.demand[j] > 0: # Only applies to warehouses with demand (sinks)
        return sum(model.flow[i, j] for i in model.Warehouses if (i, j) in model.Routes) >= model.demand[j]
    return pyo.Constraint.Skip # Skip for pure source nodes or nodes with no demand
model.demand_constraint = pyo.Constraint(model.Warehouses, rule=demand_constraint_rule)

# Note: Bounds for decision variables (min/max effective_distance_factor, speed, running_cost)
# are already handled in their respective variable definitions.

# --- Solve the model (example using a dummy solver or a real one if available) ---
# To solve this optimization model, you would typically need to install and configure a solver.
# Popular open-source solvers include GLPK and CBC. Commercial solvers include Gurobi and CPLEX.
# For example, to use GLPK (after installing it, e.g., via `apt install glpk-utils` on Linux):
# try:
#     solver = pyo.SolverFactory('glpk')
#     results = solver.solve(model, tee=True) # tee=True shows solver output

#     print("\n--- Solver Results ---")
#     model.display() # Displays all model components and their final values

#     print("\n--- Objective Value ---")
#     print(f"Total Cost: {pyo.value(model.objective):.2f}")

#     print("\n--- Decision Variable Values ---")
#     for i, j in model.Routes:
#         if pyo.value(model.flow[i, j]) > 0.001: # Only print non-zero flows for clarity
#             print(f"Flow from {i} to {j}: {pyo.value(model.flow[i, j]):.2f} units")
#         print(f"  Effective Distance Factor for {i}-{j}: {pyo.value(model.effective_distance_factor[i, j]):.2f}")
#         print(f"  Delivery Speed for {i}-{j}: {pyo.value(model.delivery_speed[i, j]):.2f}")
#     for w in model.Warehouses:
#         print(f"Warehouse Running Cost for {w}: {pyo.value(model.warehouse_running_cost[w]):.2f}")

# except Exception as e:
#     print(f"\nSolver error or GLPK not found: {e}")
#     print("Please ensure you have a solver like GLPK installed and accessible in your environment.")

print("Pyomo model created successfully. To run, uncomment the solver section and ensure a solver is installed.")

ERROR:pyomo.core:Rule failed when generating expression for Constraint supply_constraint with index W2:
PyomoException: Cannot convert non-constant Pyomo expression (0  <  (warehouse_running_cost[W2] - 500)*0.01) to bool.
This error is usually caused by using a Var, unit, or mutable Param in a
Boolean context such as an "if" statement, or when checking container
membership or equality. For example,
    >>> m.x = Var()
    >>> if m.x >= 1:
    ...     pass
and
    >>> m.y = Var()
    >>> if m.y in [m.x, m.y]:
    ...     pass
would both cause this exception.
ERROR:pyomo.core:Constructing component 'supply_constraint' from data=None failed:
    PyomoException: Cannot convert non-constant Pyomo expression (0  <  (warehouse_running_cost[W2] - 500)*0.01) to bool.
This error is usually caused by using a Var, unit, or mutable Param in a
Boolean context such as an "if" statement, or when checking container
membership or equality. For example,
    >>> m.x = Var()
    >>> if m.x >= 1:
    ...   

PyomoException: Cannot convert non-constant Pyomo expression (0  <  (warehouse_running_cost[W2] - 500)*0.01) to bool.
This error is usually caused by using a Var, unit, or mutable Param in a
Boolean context such as an "if" statement, or when checking container
membership or equality. For example,
    >>> m.x = Var()
    >>> if m.x >= 1:
    ...     pass
and
    >>> m.y = Var()
    >>> if m.y in [m.x, m.y]:
    ...     pass
would both cause this exception.